# 06a — Gradient Descent & Backpropagation

Corresponds to: Note 06 — Gradient Descent & Backpropagation

**Libraries introduced in this notebook:** No new libraries — numpy only (introduced in Note 03).

**What this notebook demonstrates:**
1. Gradient descent — update rule, loss decreasing over steps on a simple surface
2. Learning rate effect — η too large, too small, just right on the same surface
3. Batch vs SGD vs mini-batch — update counts and gradient noise on a small dataset
4. Backpropagation — full forward and backward pass on the XOR MLP from Note 03
5. Vanishing gradients — sigmoid vs ReLU across 10 layers
6. Exploding gradients and clipping — norm before and after rescaling

---
## Part 1: Gradient Descent

The update rule is: w ← w − η × ∂L/∂w

We use L(w) = (2 − w)² — minimum at w = 2, gradient ∂L/∂w = 2(w − 2).

We run gradient descent from w = −1 and watch the loss decrease toward zero.

In [ ]:
import numpy as np

def loss(w):      return (2 - w) ** 2
def grad(w):      return 2 * (w - 2)

w = -1.0
eta = 0.2

print('Gradient descent on L(w) = (2 − w)²  — minimum at w = 2:')
print(f'  {"Step":>5}   {"w":>8}   {"L(w)":>8}   {"∂L/∂w":>8}   {"update":>10}')
print(f'  {"-"*5}   {"-"*8}   {"-"*8}   {"-"*8}   {"-"*10}')
for step in range(10):
    g = grad(w)
    update = -eta * g
    print(f'  {step:>5}   {w:>8.4f}   {loss(w):>8.4f}   {g:>8.4f}   {update:>10.4f}')
    w = w + update
print(f'  {10:>5}   {w:>8.4f}   {loss(w):>8.4f}   (converged)')

---
## Part 2: Learning Rate Effect

The same loss surface L(w) = (2 − w)², starting from w = −1.

We compare three learning rates: too large (overshoots), too small (slow), just right.

In [ ]:
configs = [
    (1.2, 'too large  — overshoots'),
    (0.02, 'too small  — slow convergence'),
    (0.2,  'just right — efficient'),
]

for eta, label in configs:
    w = -1.0
    print(f'η = {eta} ({label}):')
    print(f'  {"Step":>5}   {"w":>8}   {"L(w)":>8}')
    print(f'  {"-"*5}   {"-"*8}   {"-"*8}')
    for step in range(8):
        print(f'  {step:>5}   {w:>8.4f}   {loss(w):>8.4f}')
        w = w - eta * grad(w)
    print(f'  {8:>5}   {w:>8.4f}   {loss(w):>8.4f}')
    print()

---
## Part 3: Batch vs SGD vs Mini-Batch

We simulate gradient descent on a small dataset of 8 samples.

The task: fit y = 2x (true weight = 2). Loss = MSE. We compare how many updates each method makes per epoch and how noisy the gradient is.

In [ ]:
np.random.seed(0)
n = 8
X = np.arange(1, n + 1, dtype=float)   # inputs: 1, 2, ..., 8
Y = 2 * X                               # true labels: y = 2x

def mse_grad(w, x, y):
    y_hat = w * x
    return np.mean(2 * (y_hat - y) * x)

eta = 0.01
w0  = 0.0   # start from w = 0 each time

# ── Batch GD ─────────────────────────────────────────────────────────────────
w = w0
print('Batch GD (1 update per epoch, exact gradient):')
print(f'  {"Epoch":>6}   {"w":>8}   {"gradient":>10}   {"updates so far":>15}')
print(f'  {"-"*6}   {"-"*8}   {"-"*10}   {"-"*15}')
for epoch in range(5):
    g = mse_grad(w, X, Y)
    print(f'  {epoch:>6}   {w:>8.4f}   {g:>10.4f}   {epoch:>15}')
    w -= eta * g
print(f'  Final w = {w:.4f}  (true = 2.0)\n')

# ── SGD ──────────────────────────────────────────────────────────────────────
w = w0
total_updates = 0
print('SGD (n=8 updates per epoch, noisy gradient):')
print(f'  {"Epoch":>6}   {"Sample":>7}   {"w":>8}   {"gradient":>10}')
print(f'  {"-"*6}   {"-"*7}   {"-"*8}   {"-"*10}')
for epoch in range(2):
    idx = np.random.permutation(n)
    for i in idx:
        g = mse_grad(w, X[i], Y[i])
        print(f'  {epoch:>6}   {i:>7}   {w:>8.4f}   {g:>10.4f}')
        w -= eta * g
        total_updates += 1
print(f'  Total updates after 2 epochs: {total_updates}  (batch GD would do 2)\n')

# ── Mini-batch ────────────────────────────────────────────────────────────────
w = w0
B = 4
total_updates = 0
print(f'Mini-batch GD (B={B}, {n//B} updates per epoch):')
print(f'  {"Epoch":>6}   {"Batch":>6}   {"w":>8}   {"gradient":>10}')
print(f'  {"-"*6}   {"-"*6}   {"-"*8}   {"-"*10}')
for epoch in range(3):
    idx = np.random.permutation(n)
    for b in range(n // B):
        batch = idx[b*B:(b+1)*B]
        g = mse_grad(w, X[batch], Y[batch])
        print(f'  {epoch:>6}   {b:>6}   {w:>8.4f}   {g:>10.4f}')
        w -= eta * g
        total_updates += 1
print(f'  Total updates after 3 epochs: {total_updates}  (batch GD would do 3)')

---
## Part 4: Backpropagation — XOR MLP

We reproduce the full worked example from Note 06 exactly.

Network: 2 inputs → 2 hidden neurons → 1 output. Sigmoid activations. MSE loss.

Parameters are the same as Note 03. Training sample: x = [0, 1], y = 1.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Parameters from note
W1 = np.array([[1., -1.],
               [1., -1.]])
b1 = np.array([-0.5, 1.5])
W2 = np.array([[1.], [1.]])
b2 = np.array([-1.5])
eta = 0.5

x = np.array([0., 1.])
y = 1.0

print('Parameters:')
print(f'  W1 shape: {W1.shape},  b1 shape: {b1.shape}')
print(f'  W2 shape: {W2.shape},  b2 shape: {b2.shape}')
print()

# ── Forward pass ──────────────────────────────────────────────────────────────
z1 = W1.T @ x + b1
h1 = sigmoid(z1)
z2 = W2.T @ h1 + b2
y_hat = sigmoid(z2)[0]
L = (y - y_hat) ** 2

print('Forward pass:')
print(f'  z1     = {z1}  (note: [0.5, 0.5])')
print(f'  h1     = {h1.round(4)}  (note: [0.6225, 0.6225])')
print(f'  z2     = {z2.round(4)}  (note: [-0.255])')
print(f'  y_hat  = {y_hat:.4f}  (note: 0.4365)')
print(f'  L      = {L:.4f}  (note: 0.3175)')

Now the backward pass — chain rule from output to input.

In [ ]:
# ── Backward pass ─────────────────────────────────────────────────────────────
dL_dyhat = -2 * (y - y_hat)
dL_dz2   = dL_dyhat * sigmoid_deriv(z2[0])

dL_dW2   = h1.reshape(-1, 1) * dL_dz2
dL_db2   = np.array([dL_dz2])

dL_dh1   = W2.flatten() * dL_dz2
dL_dz1   = dL_dh1 * sigmoid_deriv(z1)

dL_dW1   = np.outer(x, dL_dz1)
dL_db1   = dL_dz1

print('Backward pass:')
print(f'  dL/dyhat  = {dL_dyhat:.4f}  (note: -1.127)')
print(f'  dL/dz2    = {dL_dz2:.4f}  (note: -0.2773)')
print(f'  dL/dW2    = {dL_dW2.flatten().round(4)}  (note: [-0.1726, -0.1726])')
print(f'  dL/db2    = {dL_db2.round(4)}  (note: [-0.2773])')
print(f'  dL/dh1    = {dL_dh1.round(4)}  (note: [-0.2773, -0.2773])')
print(f'  dL/dz1    = {dL_dz1.round(4)}  (note: [-0.0652, -0.0652])')
print(f'  dL/dW1    =')
print(f'    {dL_dW1.round(4)}  (note: row0=[0,0], row1=[-0.0652,-0.0652])')
print(f'  dL/db1    = {dL_db1.round(4)}  (note: [-0.0652, -0.0652])')

Now apply the weight update and verify the new parameter values match the note.

In [ ]:
# ── Weight update ─────────────────────────────────────────────────────────────
W1_new = W1 - eta * dL_dW1
b1_new = b1 - eta * dL_db1
W2_new = W2 - eta * dL_dW2
b2_new = b2 - eta * dL_db2

print('Weight update (η = 0.5):')
print(f'  W2_new = {W2_new.flatten().round(4)}  (note: [1.0863, 1.0863])')
print(f'  b2_new = {b2_new.round(4)}  (note: [-1.3613])')
print(f'  W1_new row 0 = {W1_new[0].round(4)}  (note: [1.0, -1.0])')
print(f'  W1_new row 1 = {W1_new[1].round(4)}  (note: [1.0326, -0.9674])')
print(f'  b1_new = {b1_new.round(4)}  (note: [-0.4674, 1.5326])')
print()

# Verify: x1=0 weights unchanged
print('Weights from x1=0 unchanged (zero gradient from zero input):')
print(f'  W1[0,0]: {W1[0,0]} → {W1_new[0,0]}  (unchanged: {W1[0,0] == W1_new[0,0]})')
print(f'  W1[0,1]: {W1[0,1]} → {W1_new[0,1]}  (unchanged: {W1[0,1] == W1_new[0,1]})')

---
## Part 5: Vanishing Gradients

Backpropagation multiplies a derivative factor at each layer. For sigmoid, this factor is at most 0.25. For ReLU (z > 0), it is exactly 1.

We show what happens to a gradient of 1.0 as it passes through 10 layers.

In [ ]:
print('Vanishing gradient — gradient reaching each layer (starting from 1.0 at output):')
print(f'  {"Layer":>7}   {"Sigmoid (×0.25 each)": >24}   {"ReLU (×1.0 each)":>20}')
print(f'  {"-"*7}   {"-"*24}   {"-"*20}')

g_sigmoid = 1.0
g_relu    = 1.0
for layer in range(1, 11):
    g_sigmoid *= 0.25
    g_relu    *= 1.0
    print(f'  {layer:>7}   {g_sigmoid:>24.10f}   {g_relu:>20.6f}')

print()
print(f'After 10 sigmoid layers: gradient = {0.25**10:.2e}')
print(f'After 10 ReLU layers:    gradient = {1.0**10:.2e}')
print()
print('Sigmoid gradient is ~10^6 times smaller — early layers receive no signal.')

---
## Part 6: Exploding Gradients and Clipping

If each layer contributes a factor greater than 1, gradients grow exponentially.

Gradient clipping rescales the gradient vector when its norm exceeds a threshold — preserving direction, reducing magnitude.

In [ ]:
# Exploding gradient across 10 layers with factor 2
print('Exploding gradient — factor 2.0 per layer:')
print(f'  {"Layer":>7}   {"Gradient magnitude":>20}')
print(f'  {"-"*7}   {"-"*20}')
g = 1.0
for layer in range(1, 11):
    g *= 2.0
    print(f'  {layer:>7}   {g:>20.1f}')
print()

# Gradient clipping example from note
gradient = np.array([500., 300., 800.])
threshold = 1.0
norm = np.linalg.norm(gradient)

print('Gradient clipping (threshold = 1.0):')
print(f'  Original gradient:  {gradient}')
print(f'  Norm:               {norm:.1f}  (note: ~990)')
print()

if norm > threshold:
    clipped = gradient * (threshold / norm)
else:
    clipped = gradient

print(f'  Clipped gradient:   {clipped.round(3)}  (note: [0.505, 0.303, 0.808])')
print(f'  Clipped norm:       {np.linalg.norm(clipped):.4f}  (should be 1.0)')
print()
print('Direction preserved:')
orig_unit   = gradient / norm
clipped_unit = clipped / np.linalg.norm(clipped)
print(f'  Original unit vector: {orig_unit.round(4)}')
print(f'  Clipped unit vector:  {clipped_unit.round(4)}')
print(f'  Same direction: {np.allclose(orig_unit, clipped_unit)}')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| Gradient descent | w converges to minimum at w=2 — loss decreases each step |
| Learning rate too large | Overshoots — loss oscillates or grows |
| Learning rate too small | Converges but very slowly |
| Batch vs SGD vs mini-batch | Batch: 1 update/epoch; SGD: n updates/epoch; mini-batch: n/B updates/epoch |
| Backprop forward pass | z1=[0.5,0.5], h1=[0.6225,0.6225], ŷ=0.4365, L=0.3175 — matches note |
| Backprop backward pass | All gradients match note; x1=0 weights receive zero gradient |
| Weight update | All updated values match note exactly |
| Vanishing gradient | Sigmoid: 0.25^10 ≈ 10^-6 after 10 layers; ReLU: 1.0 — no shrinkage |
| Exploding gradient | Factor 2^10 = 1024 after 10 layers |
| Gradient clipping | Norm rescaled to threshold — direction preserved exactly |